In [2]:
# For RAG, we used the A->Q->A' setup:

# A = original answer in the FAQ
# Q = generated question from this answer
# A' = answer produced by our RAG system
# For agents, we use the same setup. A' comes from an agent instead of a fixed RAG pipeline.

# We also save the trajectory. Here, the trajectory means only the tool calls the agent made before producing the final answer.

# Loading the data
# Use the same ground truth questions:

import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
# Load the FAQ documents and the search index:

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# Create a lookup table:

doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
# Running the agent
# Reuse the ToyAIKit agent from module 01. It handles the agent loop and stores the full message history.

# First, set up the model clients:

from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

# Define the search tool:

def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

# Create the runner:

from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)



In [7]:
# The result contains:

# -- last_message: the final response
# -- all_messages: the full message history
# -- cost: the cost of all LLM calls in this run

# Run it for one ground truth question:

rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

# Look at the full message history:

result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Can I still join the course if I just found it now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"late join course found it now can I still join"}', call_id='call_DbcsSd9NTou7Iie8AwVek3Kd', name='search', type='function_call', id='fc_051badde8366567c006a53ee5067e88192b34fe52ad720ff56', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_DbcsSd9NTou7Iie8AwVek3Kd',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting 

In [8]:
# For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

# Extract the function name and arguments:

def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [9]:
# For this example:

tool_calls = extract_tool_calls(result.all_messages)

tool_calls

# You should see something like this:

# [
#     {
#         "name": "search",
#         "arguments": "{\"query\":\"own pace certificate at the end self-paced course certificate\"}"
#     }
# ]

[{'name': 'search',
  'arguments': '{"query":"late join course found it now can I still join"}'}]

In [10]:
# Get the original answer:

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

# Save the A->Q->A' record and the trajectory:

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

# The answer_agent field is what we evaluate with the LLM judge. 
# The tool_calls field lets the judge see how the agent got there.

{'question': 'Can I still join the course if I just found it now?',
 'answer_agent': 'Yes — you can still join the course if you just discovered it now.\n\nIf you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"late join course found it now can I still join"}'}],
 'cost': Decimal('0.00093225'),
 'document': '74eb249bbf'}